# 01 — EDA e baseline (Fase 2)
Runtime: **CPU padrão** (não precisa de GPU). Execute as células em ordem.

Objetivos: entender os padrões de fraude, decidir o tipo de split e medir um baseline (Regressão Logística)
com e sem os saldos pós-transação.

In [ ]:
import os, sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/fraud-detection-paysim'
except ImportError:
    PROJECT_DIR = os.getcwd()
os.chdir(PROJECT_DIR)
sys.path.insert(0, os.path.join(PROJECT_DIR, 'src'))

import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print

from fraud.config import load_config
cfg = load_config()
y = cfg['data']['target']
FIG = os.path.join(cfg['paths']['reports_dir'], 'figures')
os.makedirs(FIG, exist_ok=True)
pd.options.display.float_format = '{:,.4f}'.format
print('Projeto:', PROJECT_DIR)

## 1. Carga dos dados
Se o Parquet ainda estiver com precisão float32 (versão anterior), ele é reconstruído com float64.

In [ ]:
from fraud.data import csv_to_parquet, load_parquet

pq = cfg['paths']['parquet']
rebuild = (not os.path.exists(pq)) or (load_parquet(pq)['amount'].dtype != 'float64')
if rebuild:
    print('Reconstruindo Parquet com float64 (1–3 min)...')
    csv_to_parquet(cfg['paths']['raw_csv'], pq)
df = load_parquet(pq)
print(df.shape, df['amount'].dtype)
print(f"Fraudes: {int(df[y].sum()):,} ({100*df[y].mean():.4f}%)")

## 2. Fraude por tipo de transação

In [ ]:
tipo = df.groupby('type', observed=True).agg(transacoes=(y, 'size'), fraudes=(y, 'sum'))
tipo['taxa_fraude_%'] = 100 * tipo['fraudes'] / tipo['transacoes']
tipo['valor_fraudado'] = df[df[y] == 1].groupby('type', observed=True)['amount'].sum()
tipo['valor_fraudado'] = tipo['valor_fraudado'].fillna(0)
display(tipo)

## 3. Valor das transações: fraude x legítima (TRANSFER e CASH_OUT)

In [ ]:
alvo = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()
fig, ax = plt.subplots(figsize=(8, 4))
bins = np.logspace(0, 8.5, 60)
for label, g in alvo.groupby(y):
    vals = g['amount'].clip(lower=1)
    ax.hist(vals, bins=bins, weights=np.full(len(vals), 100 / len(vals)), alpha=0.55,
            label='fraude' if label == 1 else 'legítima')
ax.set_xscale('log'); ax.set_xlabel('valor (escala log)'); ax.set_ylabel('% das transações do grupo')
ax.set_title('Valor da transação — TRANSFER e CASH_OUT'); ax.legend()
fig.tight_layout(); fig.savefig(os.path.join(FIG, 'valor_fraude_vs_legitima.png'), dpi=130); plt.show()
print('Fraudes com valor 0:', int(((alvo[y] == 1) & (alvo['amount'] == 0)).sum()))
print('Fraudes com valor exato de 10 milhões:', int(((alvo[y] == 1) & (alvo['amount'] == 10_000_000)).sum()))
display(alvo.groupby(y)['amount'].describe(percentiles=[.5, .9, .99]))

## 4. Comportamento no tempo
O PaySim tem muito mais transações nas primeiras horas, mas fraudes ao longo de todo o período.

In [ ]:
por_step = df.groupby('step').agg(transacoes=(y, 'size'), fraudes=(y, 'sum'))
fig, ax = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
ax[0].plot(por_step.index, por_step['transacoes']); ax[0].set_ylabel('transações')
ax[1].plot(por_step.index, por_step['fraudes'], color='tab:red'); ax[1].set_ylabel('fraudes')
ax[1].set_xlabel('step (hora simulada)')
fig.tight_layout(); fig.savefig(os.path.join(FIG, 'serie_temporal.png'), dpi=130); plt.show()

corte = por_step.index.min() + int(0.25 * len(por_step))
share = por_step.loc[por_step.index <= corte, 'transacoes'].sum() / por_step['transacoes'].sum()
print(f"{share:.1%} das transações estão nos primeiros 25% dos steps")

## 5. Padrões de saldo (indícios de vazamento do simulador)

In [ ]:
alvo['esvazia_origem'] = (alvo['newbalanceOrig'] == 0) & (alvo['oldbalanceOrg'] > 0)
alvo['valor_igual_saldo'] = np.isclose(alvo['amount'], alvo['oldbalanceOrg'])
alvo['destino_saldo_zero'] = alvo['oldbalanceDest'] == 0
pad = alvo.groupby(y)[['esvazia_origem', 'valor_igual_saldo', 'destino_saldo_zero']].mean() * 100
pad.columns = [c + ' (%)' for c in pad.columns]
display(pad)

alvo['erro_origem'] = alvo['newbalanceOrig'] + alvo['amount'] - alvo['oldbalanceOrg']
alvo['erro_destino'] = alvo['oldbalanceDest'] + alvo['amount'] - alvo['newbalanceDest']
tol = 0.01
erros = alvo.groupby(y).apply(lambda g: pd.Series({
    'erro_origem != 0 (%)': (g['erro_origem'].abs() > tol).mean() * 100,
    'erro_destino != 0 (%)': (g['erro_destino'].abs() > tol).mean() * 100,
}))
display(erros)

flag = df.groupby('isFlaggedFraud')[y].agg(['size', 'sum'])
print('Regra do simulador (isFlaggedFraud) — não será usada como feature:')
display(flag)

## 6. Qual split usar?
Comparação entre cortar por **intervalo de steps** (atual) e por **número de transações**.

In [ ]:
from fraud.split import temporal_split, temporal_split_by_rows

s = cfg['split']
splits = {
    'por_steps': temporal_split(df, s['train_end'], s['val_end']),
    'por_linhas': temporal_split_by_rows(df, s['train_end'], s['val_end']),
}
linhas = []
for nome, partes in splits.items():
    for conj, p in zip(['treino', 'validação', 'teste'], partes):
        linhas.append({'split': nome, 'conjunto': conj, 'linhas': len(p),
                       'fraudes': int(p[y].sum()), 'taxa_fraude_%': 100 * p[y].mean(),
                       'steps': f"{p['step'].min()}–{p['step'].max()}"})
comp = pd.DataFrame(linhas)
display(comp)
comp.to_csv(os.path.join(cfg['paths']['reports_dir'], 'comparacao_splits.csv'), index=False)

## 7. Baseline: Regressão Logística (pesos de classe)
Treino em amostra (todas as fraudes + 15% das legítimas) só para ser rápido; a avaliação usa a **validação completa**.
O conjunto de teste não é tocado nesta etapa. Rodamos os dois tipos de split e as duas variantes de features.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_recall_curve
from fraud.data import dev_sample
from fraud.features import build_features
from fraud.metrics import evaluate

resultados, curvas = [], {}
for nome_split, (train, val, test) in splits.items():
    train_s = dev_sample(train, cfg['data']['sample_frac_dev'], target=y, seed=cfg['seed'])
    for v in cfg['features']['variants'][:2]:
        Xtr, ytr = build_features(train_s, v), train_s[y]
        Xva, yva = build_features(val, v), val[y]
        modelo = make_pipeline(StandardScaler(),
                               LogisticRegression(class_weight='balanced', max_iter=1000))
        modelo.fit(Xtr, ytr)
        score = modelo.predict_proba(Xva)[:, 1]
        r = evaluate(yva.values, score, amount=val['amount'].values)
        r.update({'split': nome_split, 'variante': v})
        resultados.append(r)
        curvas[(nome_split, v)] = precision_recall_curve(yva.values, score)[:2]
        print('ok', nome_split, v)

res = pd.DataFrame(resultados).set_index(['split', 'variante'])
cols = ['prevalencia_%', 'pr_auc', 'roc_auc', 'recall@precision>=90%',
        'bestF1_precision', 'bestF1_recall', 'bestF1_f1', 'bestF1_fpr', 'bestF1_value_recall']
display(res[cols])
res.to_csv(os.path.join(cfg['paths']['reports_dir'], 'baseline_logreg_validacao.csv'))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for a, nome_split in zip(ax, splits):
    for v in cfg['features']['variants'][:2]:
        p, r = curvas[(nome_split, v)]
        a.plot(r, p, label=v)
    a.set_title(f'Curva Precision x Recall — {nome_split}')
    a.set_xlabel('recall'); a.legend()
ax[0].set_ylabel('precision')
fig.tight_layout(); fig.savefig(os.path.join(FIG, 'baseline_pr_curves.png'), dpi=130); plt.show()

## O que enviar de volta
Copie e cole (ou tire print) das saídas das seções **2, 5, 6 e 7**. Com elas decidimos o split e fechamos as metas do baseline.